# Individual Differences in Predictability

**Purpose**: Investigate why some subjects are highly predictable (~90% accuracy) while others are near chance. Correlate subject-level predictability with behavioral traits.

**Key Questions**:
1. What is the distribution of subject-level prediction accuracy?
2. Who are the highly predictable vs. unpredictable subjects?
3. Do behavioral traits (RT variability, choice consistency, ambiguity sensitivity) explain predictability?
4. Are there distinct "types" of decision-makers?

**Why This Matters**:
- Understanding individual differences helps identify who benefits most from physiological monitoring
- Reveals whether predictability reflects decision style, data quality, or both
- Informs personalized BCI/neurofeedback applications

In [37]:
import sys
sys.path.append('../..')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Paths
OUTPUT_DIR = Path('../../data/results/main/statistical_analyses/individual_differences')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'='*70}")
print(f"INDIVIDUAL DIFFERENCES IN PREDICTABILITY")
print(f"{'='*70}")
print(f"Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

INDIVIDUAL DIFFERENCES IN PREDICTABILITY
Analysis started: 2026-06-04 09:43:12


## 1. Load Data

In [ ]:
# Load main features
with open('../../data/features/extracted_features_PRE.pkl', 'rb') as f:
    feature_data = pickle.load(f)

merged_df     = feature_data['merged_df'].copy()
physio_cols   = feature_data['physio_cols']
gaze_cols     = feature_data['gaze_cols']
behavior_cols = feature_data['behavior_cols']

# Load and merge EEG features (regional_bands, 16 features — same file as fusion notebooks 03)
with open('../../data/features/eeg_features_regional_bands.pkl', 'rb') as f:
    eeg_data = pickle.load(f)
eeg_df   = eeg_data['eeg_features_df']
eeg_cols = eeg_data['feature_columns']
merged_df = merged_df.merge(eeg_df[['trial_id'] + eeg_cols],
                             on='trial_id', how='inner')
print(f"EEG features merged: {len(eeg_cols)} cols, {eeg_df['subject_id'].nunique()} subjects with EEG")

# Add visit info from session mapping
sm = pd.read_csv('../../data/results/session_mapping.csv',
                 dtype={'mmdd': str, 'hhmm': str, 'user_id': str})
sm['subject_id'] = sm['mmdd'] + '_' + sm['hhmm'] + '_' + sm['user_id']
merged_df = merged_df.merge(sm[['subject_id', 'visit_number', 'team']], on='subject_id', how='left')

# Full feature set: behavior + gaze + physio + EEG
all_feature_cols = behavior_cols + gaze_cols + physio_cols + eeg_cols
all_feature_cols = [c for c in all_feature_cols if c in merged_df.columns]

print(f"Loaded {len(merged_df)} trials from {merged_df['subject_id'].nunique()} subjects (EEG set)")
print(f"\nFeature counts:")
print(f"  Behavior:  {len(behavior_cols)}")
print(f"  Gaze:      {len(gaze_cols)}")
print(f"  Physio:    {len(physio_cols)}")
print(f"  EEG:       {len([c for c in eeg_cols if c in merged_df.columns])}")
print(f"  Total:     {len(all_feature_cols)}")
print(f"\nVisit breakdown:")
print(merged_df.groupby('visit_number')['subject_id'].nunique())


## 2. Compute Subject-Level Prediction Accuracy

In [ ]:
def get_subject_prediction_accuracy(df, feature_cols, feature_name='features'):
    """Run LOSO-CV and return per-subject accuracy."""
    logo = LeaveOneGroupOut()
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()

    X = df[feature_cols].values
    y = df['outcome'].values
    subjects = df['subject_id'].values

    subject_results = []

    for train_idx, test_idx in logo.split(X, y, subjects):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        subj = subjects[test_idx[0]]

        X_train = imputer.fit_transform(X_train)
        X_test  = imputer.transform(X_test)
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        clf = RandomForestClassifier(n_estimators=100, max_depth=5,
                                     class_weight='balanced', random_state=42, n_jobs=-1)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob) if len(np.unique(y_test)) > 1 else np.nan

        subject_results.append({
            'subject_id': subj,
            f'{feature_name}_accuracy': acc,
            f'{feature_name}_auc': auc,
            'n_trials': len(y_test),
            'invest_rate': y_test.mean()
        })

    return pd.DataFrame(subject_results)

print("Computing subject-level prediction accuracies...")
print("(This may take a few minutes)\n")

# Full model (behavior + gaze + physio + EEG)
print("Running full model prediction...")
full_model_results = get_subject_prediction_accuracy(merged_df, all_feature_cols, 'full_model')

# Behavior only (for comparison / scatter)
print("Running behavior-only prediction...")
behavior_results = get_subject_prediction_accuracy(merged_df, behavior_cols, 'behavior')

# Merge
subject_acc_df = full_model_results.merge(
    behavior_results[['subject_id', 'behavior_accuracy', 'behavior_auc']],
    on='subject_id'
)

print(f"\nComputed accuracies for {len(subject_acc_df)} subjects")

In [ ]:
print("\n" + "="*70)
print("DISTRIBUTION OF SUBJECT-LEVEL PREDICTION ACCURACY")
print("="*70)

print(f"\nFull Model Accuracy:")
print(f"  Mean:  {subject_acc_df['full_model_accuracy'].mean():.3f}")
print(f"  Std:   {subject_acc_df['full_model_accuracy'].std():.3f}")
print(f"  Min:   {subject_acc_df['full_model_accuracy'].min():.3f}")
print(f"  Max:   {subject_acc_df['full_model_accuracy'].max():.3f}")
print(f"  Range: {subject_acc_df['full_model_accuracy'].max() - subject_acc_df['full_model_accuracy'].min():.3f}")

subject_acc_df['predictability_group'] = pd.cut(
    subject_acc_df['full_model_accuracy'],
    bins=[0, 0.55, 0.65, 0.75, 1.0],
    labels=['Near Chance (<55%)', 'Low (55-65%)', 'Medium (65-75%)', 'High (>75%)']
)

print(f"\nPredictability Groups:")
print(subject_acc_df['predictability_group'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Histogram
ax = axes[0]
ax.hist(subject_acc_df['full_model_accuracy'], bins=20, color='#2E86AB', alpha=0.7, edgecolor='black')
ax.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Chance')
ax.axvline(subject_acc_df['full_model_accuracy'].mean(), color='green', linestyle='-', linewidth=2, label='Mean')
ax.set_xlabel('Prediction Accuracy')
ax.set_ylabel('Number of Subjects')
ax.set_title('Distribution of Subject-Level Accuracy\n(Full Model)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Sorted bar chart
ax = axes[1]
sorted_df = subject_acc_df.sort_values('full_model_accuracy', ascending=False)
colors = ['#E74C3C' if acc < 0.55 else '#F39C12' if acc < 0.65 else '#27AE60' if acc < 0.75 else '#2E86AB'
          for acc in sorted_df['full_model_accuracy']]
ax.bar(range(len(sorted_df)), sorted_df['full_model_accuracy'], color=colors, alpha=0.8)
ax.axhline(0.5, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Subject (sorted by accuracy)')
ax.set_ylabel('Prediction Accuracy')
ax.set_title('Subject-Level Accuracy (Sorted)\nRed=<55%, Orange=55-65%, Green=65-75%, Blue=>75%', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Full model vs behavior-only accuracy
ax = axes[2]
ax.scatter(subject_acc_df['behavior_accuracy'], subject_acc_df['full_model_accuracy'],
           c='#2E86AB', alpha=0.6, s=50)
ax.plot([0.4, 1], [0.4, 1], 'k--', alpha=0.5, label='y=x')
ax.axhline(0.5, color='red', linestyle=':', alpha=0.5)
ax.axvline(0.5, color='red', linestyle=':', alpha=0.5)
r, p = stats.pearsonr(subject_acc_df['behavior_accuracy'], subject_acc_df['full_model_accuracy'])
ax.set_xlabel('Behavior-Only Accuracy')
ax.set_ylabel('Full Model Accuracy')
ax.set_title(f'Full Model vs Behavior-Only Accuracy\n(r = {r:.3f}, p = {p:.2e})', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'subject_accuracy_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Compute Behavioral Traits for Each Subject

In [ ]:
# Compute subject-level behavioral traits using behavior features from the pickle
print("\n" + "="*70)
print("COMPUTING BEHAVIORAL TRAITS PER SUBJECT")
print("="*70)

print(f"\nBehavior features available: {behavior_cols}")

subject_traits = []

for subj in merged_df['subject_id'].unique():
    subj_df = merged_df[merged_df['subject_id'] == subj]
    
    # Basic stats
    n_trials = len(subj_df)
    invest_rate = subj_df['outcome'].mean()
    
    # === RT statistics (from behavior features) ===
    rt_mean = subj_df['decision_time'].mean()
    rt_std = subj_df['decision_time'].std()
    rt_cv = rt_std / rt_mean if rt_mean > 0 else np.nan  # Coefficient of variation
    rt_median = subj_df['decision_time'].median()
    
    # === Use behavior features from pickle ===
    # EV difference variability - how variable are their EV differences across trials
    ev_diff_mean = subj_df['ev_difference'].mean()
    ev_diff_std = subj_df['ev_difference'].std()
    
    # Risk premium variability
    risk_premium_mean = subj_df['risk_premium'].mean()
    risk_premium_std = subj_df['risk_premium'].std()
    
    # Investment variance sensitivity (average across trials)
    invest_variance_mean = subj_df['invest_variance'].mean()
    
    # Sequential behavior features (averaged per subject)
    running_invest_rate_mean = subj_df['running_invest_rate'].mean()
    running_invest_rate_std = subj_df['running_invest_rate'].std()
    recent_invest_rate_std = subj_df['recent_invest_rate_5'].std()  # Variability in recent choices
    
    # === Derived behavioral traits ===
    # Choice consistency - how often do they make the same choice for similar ambiguity?
    choice_consistency = []
    for amb in [0, 3, 6]:
        amb_df = subj_df[subj_df['ambiguity'] == amb]
        if len(amb_df) > 1:
            # Consistency = how far from 50% is their invest rate
            consistency = abs(amb_df['outcome'].mean() - 0.5) * 2  # Ranges 0-1
            choice_consistency.append(consistency)
    mean_choice_consistency = np.mean(choice_consistency) if choice_consistency else np.nan
    
    # Ambiguity sensitivity - how much does ambiguity affect choice?
    low_amb_invest = subj_df[subj_df['ambiguity'] == 0]['outcome'].mean()
    high_amb_invest = subj_df[subj_df['ambiguity'] == 6]['outcome'].mean()
    ambiguity_sensitivity = abs(low_amb_invest - high_amb_invest) if not np.isnan(low_amb_invest) and not np.isnan(high_amb_invest) else np.nan
    
    # RT by choice - do they deliberate more for one choice?
    invest_rt = subj_df[subj_df['outcome'] == 1]['decision_time'].mean()
    keep_rt = subj_df[subj_df['outcome'] == 0]['decision_time'].mean()
    rt_choice_diff = invest_rt - keep_rt if not np.isnan(invest_rt) and not np.isnan(keep_rt) else np.nan
    
    # Social condition proportion
    social_prop = subj_df['condition_social'].mean()
    
    # === Physiological variability ===
    pupil_mean_var = subj_df['pupil_mean_pre'].std() if 'pupil_mean_pre' in subj_df.columns else np.nan
    gaze_x_std_var = subj_df['gaze_x_std'].std() if 'gaze_x_std' in subj_df.columns else np.nan
    
    subject_traits.append({
        'subject_id': subj,
        'n_trials': n_trials,
        'subject_invest_rate': invest_rate,  # Renamed to avoid conflict
        # RT features
        'rt_mean': rt_mean,
        'rt_std': rt_std,
        'rt_cv': rt_cv,
        'rt_median': rt_median,
        'rt_choice_diff': rt_choice_diff,
        # Behavior features from pickle
        'ev_diff_mean': ev_diff_mean,
        'ev_diff_std': ev_diff_std,
        'risk_premium_mean': risk_premium_mean,
        'risk_premium_std': risk_premium_std,
        'invest_variance_mean': invest_variance_mean,
        'social_prop': social_prop,
        # Sequential behavior
        'running_invest_rate_std': running_invest_rate_std,
        'recent_invest_rate_std': recent_invest_rate_std,
        # Derived traits
        'choice_consistency': mean_choice_consistency,
        'ambiguity_sensitivity': ambiguity_sensitivity,
        # Physiological variability
        'pupil_variability': pupil_mean_var,
        'gaze_variability': gaze_x_std_var
    })

traits_df = pd.DataFrame(subject_traits)
print(f"\nComputed {len(traits_df.columns)-1} traits for {len(traits_df)} subjects")
print(f"\nTraits: {list(traits_df.columns[1:])}")

## 3b. Ambiguity Aversion & Gaze Coupling Scores

Following notebook 14b: per-subject ambiguity aversion (`invest_rate(amb=0) − invest_rate(amb>0)`) and gaze coupling (mean |β| of top gaze features regressed on ambiguity). These are added as traits alongside the behavioral features above.

In [ ]:
# Compute ambiguity aversion and gaze coupling scores (from notebook 14b)
COUPLING_FEATURES = ['gaze_y_std', 'gaze_y_mean', 'gaze_x_std']

def compute_aversion_coupling(trials):
    if len(trials) < 20:
        return None
    r0   = trials[trials['ambiguity'] == 0]['outcome'].mean()
    rpos = trials[trials['ambiguity'] >  0]['outcome'].mean()
    if pd.isna(r0) or pd.isna(rpos):
        return None
    aversion = r0 - rpos

    amb = trials['ambiguity'].values
    amb_z = (amb - amb.mean()) / (amb.std() + 1e-9)
    coupling_betas = {}
    for col in COUPLING_FEATURES:
        if col not in trials.columns:
            continue
        common_idx = trials[col].dropna().index.intersection(trials.index)
        if len(common_idx) < 10:
            continue
        f_vals = trials.loc[common_idx, col].values
        a_vals = amb_z[trials.index.get_indexer(common_idx)]
        if a_vals.std() == 0:
            continue
        beta = np.cov(f_vals, a_vals)[0, 1] / np.var(a_vals)
        coupling_betas[col] = beta
    if not coupling_betas:
        return None
    coupling_score = np.mean(np.abs(list(coupling_betas.values())))
    return {'aversion_score': aversion, 'gaze_coupling_score': coupling_score}

aversion_rows = []
for subj, grp in merged_df.groupby('subject_id'):
    res = compute_aversion_coupling(grp)
    if res is not None:
        aversion_rows.append({'subject_id': subj, **res})

aversion_df = pd.DataFrame(aversion_rows)
print(f"Aversion/coupling computed for {len(aversion_df)} subjects")
print(aversion_df[['aversion_score', 'gaze_coupling_score']].describe().round(3))

In [ ]:
# Merge traits with accuracy, aversion scores, and visit info
subject_acc_clean = subject_acc_df.drop(columns=['invest_rate'], errors='ignore')
analysis_df = subject_acc_clean.merge(traits_df, on='subject_id')
analysis_df = analysis_df.merge(aversion_df, on='subject_id', how='left')
visit_info = merged_df[['subject_id', 'visit_number', 'team']].drop_duplicates('subject_id')
analysis_df = analysis_df.merge(visit_info, on='subject_id', how='left')

print(f"\nMerged dataset: {len(analysis_df)} subjects")
print(f"\nTrait summary statistics:")
print(analysis_df[['rt_cv', 'choice_consistency', 'ambiguity_sensitivity',
                    'subject_invest_rate', 'aversion_score', 'gaze_coupling_score']].describe().round(3))

## 4. Correlate Predictability with Behavioral Traits

In [ ]:
print("\n" + "="*70)
print("CORRELATIONS: PREDICTABILITY vs BEHAVIORAL TRAITS")
print("="*70)

trait_cols = ['rt_mean', 'rt_std', 'rt_cv', 'rt_choice_diff',
              'ev_diff_mean', 'ev_diff_std', 'risk_premium_mean', 'risk_premium_std',
              'invest_variance_mean', 'social_prop',
              'running_invest_rate_std', 'recent_invest_rate_std',
              'choice_consistency', 'ambiguity_sensitivity', 'subject_invest_rate',
              'pupil_variability', 'gaze_variability', 'n_trials',
              'aversion_score', 'gaze_coupling_score']

correlations = []
print("\nCorrelations with full-model prediction accuracy:")
print("-" * 60)

for trait in trait_cols:
    if trait not in analysis_df.columns:
        continue
    valid = analysis_df[['full_model_accuracy', trait]].dropna()
    if len(valid) > 10:
        r, p = stats.pearsonr(valid['full_model_accuracy'], valid[trait])
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        print(f"  {trait:25s}: r = {r:+.3f}, p = {p:.3e} {sig}")
        correlations.append({'trait': trait, 'r': r, 'p': p, 'n': len(valid)})

corr_df = pd.DataFrame(correlations).sort_values('p')
print("\n" + "-" * 60)
print("Top predictors of subject-level accuracy (sorted by p-value):")
print(corr_df.head(5).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

key_traits = ['rt_cv', 'choice_consistency', 'ambiguity_sensitivity',
              'subject_invest_rate', 'running_invest_rate_std', 'gaze_variability']
trait_labels = ['RT Variability (CV)', 'Choice Consistency', 'Ambiguity Sensitivity',
                'Invest Rate', 'Running Invest Rate Std', 'Gaze Variability']

for idx, (trait, label) in enumerate(zip(key_traits, trait_labels)):
    ax = axes[idx // 3, idx % 3]

    if trait not in analysis_df.columns:
        ax.set_visible(False)
        continue

    valid = analysis_df[['full_model_accuracy', trait]].dropna()
    ax.scatter(valid[trait], valid['full_model_accuracy'], c='#2E86AB', alpha=0.6, s=50)

    if len(valid) > 10:
        z = np.polyfit(valid[trait], valid['full_model_accuracy'], 1)
        x_line = np.linspace(valid[trait].min(), valid[trait].max(), 100)
        ax.plot(x_line, np.poly1d(z)(x_line), 'r-', alpha=0.7, linewidth=2)
        r, pval = stats.pearsonr(valid[trait], valid['full_model_accuracy'])
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        ax.set_title(f'{label}\nr = {r:.3f} ({sig})', fontweight='bold')

    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel(label)
    ax.set_ylabel('Full Model Accuracy')
    ax.grid(True, alpha=0.3)

plt.suptitle('Behavioral Traits vs Full Model Prediction Accuracy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'traits_vs_predictability.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Characterize High vs Low Predictability Subjects

In [ ]:
print("\n" + "="*70)
print("COMPARING HIGH vs LOW PREDICTABILITY SUBJECTS")
print("="*70)

q75 = analysis_df['full_model_accuracy'].quantile(0.75)
q25 = analysis_df['full_model_accuracy'].quantile(0.25)

high_pred = analysis_df[analysis_df['full_model_accuracy'] >= q75]
low_pred  = analysis_df[analysis_df['full_model_accuracy'] <= q25]

print(f"\nHigh predictability (top 25%): {len(high_pred)} subjects, acc >= {q75:.3f}")
print(f"Low predictability (bottom 25%): {len(low_pred)} subjects, acc <= {q25:.3f}")

print("\n" + "-" * 60)
print("Trait comparison (High vs Low predictability):")
print("-" * 60)

comparison_results = []
for trait in trait_cols:
    if trait not in analysis_df.columns:
        continue
    high_vals = high_pred[trait].dropna()
    low_vals  = low_pred[trait].dropna()
    if len(high_vals) > 5 and len(low_vals) > 5:
        t_stat, p_val = stats.ttest_ind(high_vals, low_vals)
        pooled_std = np.sqrt(((len(high_vals)-1)*high_vals.var() + (len(low_vals)-1)*low_vals.var()) /
                             (len(high_vals) + len(low_vals) - 2))
        d = (high_vals.mean() - low_vals.mean()) / pooled_std if pooled_std > 0 else 0
        sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else ''))
        print(f"  {trait:25s}: High={high_vals.mean():.3f}, Low={low_vals.mean():.3f}, d={d:+.2f}, p={p_val:.3f} {sig}")
        comparison_results.append({
            'trait': trait, 'high_mean': high_vals.mean(), 'low_mean': low_vals.mean(),
            'cohens_d': d, 'p_value': p_val
        })

comparison_df = pd.DataFrame(comparison_results)

In [ ]:
# Visualization: High vs Low predictability profiles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Effect sizes for each trait
ax = axes[0]
sorted_comp = comparison_df.sort_values('cohens_d', ascending=True)
colors = ['#E74C3C' if d < 0 else '#27AE60' for d in sorted_comp['cohens_d']]
ax.barh(range(len(sorted_comp)), sorted_comp['cohens_d'], color=colors, alpha=0.8)
ax.set_yticks(range(len(sorted_comp)))
ax.set_yticklabels(sorted_comp['trait'])
ax.axvline(0, color='black', linewidth=0.5)
ax.axvline(-0.2, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0.2, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel("Cohen's d (High - Low predictability)")
ax.set_title('Trait Differences: High vs Low Predictability\n(Green = higher in High pred, Red = higher in Low pred)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Plot 2: Box plots for key differentiating traits
ax = axes[1]
# Get top 2 differentiating traits by absolute effect size
top_traits = comparison_df.reindex(comparison_df['cohens_d'].abs().sort_values(ascending=False).index).head(2)['trait'].tolist()

plot_data = []
for trait in top_traits:
    for _, row in high_pred.iterrows():
        plot_data.append({'Group': 'High Pred', 'Trait': trait, 'Value': row[trait]})
    for _, row in low_pred.iterrows():
        plot_data.append({'Group': 'Low Pred', 'Trait': trait, 'Value': row[trait]})

plot_df = pd.DataFrame(plot_data)
sns.boxplot(data=plot_df, x='Trait', y='Value', hue='Group', 
            palette={'High Pred': '#27AE60', 'Low Pred': '#E74C3C'}, ax=ax)
ax.set_title('Top Differentiating Traits', fontweight='bold')
ax.legend(title='Predictability')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'high_vs_low_predictability.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Multiple Regression: What Predicts Predictability?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

print("\n" + "="*70)
print("MULTIPLE REGRESSION: PREDICTING SUBJECT PREDICTABILITY")
print("="*70)

predictor_cols = ['rt_cv', 'choice_consistency', 'ambiguity_sensitivity',
                  'subject_invest_rate', 'running_invest_rate_std',
                  'n_trials', 'gaze_variability',
                  'aversion_score', 'gaze_coupling_score']
predictor_cols = [c for c in predictor_cols if c in analysis_df.columns]

reg_df = analysis_df[['full_model_accuracy'] + predictor_cols].dropna()
print(f"\nUsing {len(reg_df)} subjects with complete data")
print(f"Predictors: {predictor_cols}")

X = reg_df[predictor_cols].values
y = reg_df['full_model_accuracy'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

reg = LinearRegression()
reg.fit(X_scaled, y)
cv_scores = cross_val_score(reg, X_scaled, y, cv=5, scoring='r2')

print(f"\nModel Performance:")
print(f"  R² (full data): {reg.score(X_scaled, y):.3f}")
print(f"  R² (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

coef_df = pd.DataFrame({
    'Predictor': predictor_cols,
    'Coefficient': reg.coef_,
    'Abs_Coefficient': np.abs(reg.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print(f"\nStandardized Coefficients:")
for _, row in coef_df.iterrows():
    print(f"  {row['Predictor']:25s}: {'+' if row['Coefficient'] > 0 else ''}{row['Coefficient']:.3f}")

In [ ]:
# Visualization: Regression coefficients
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#27AE60' if c > 0 else '#E74C3C' for c in coef_df['Coefficient']]
ax.barh(range(len(coef_df)), coef_df['Coefficient'], color=colors, alpha=0.8)
ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df['Predictor'])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Standardized Coefficient')
ax.set_title(f'What Predicts Subject Predictability?\n(R² = {reg.score(X_scaled, y):.3f})', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add text annotation
ax.text(0.95, 0.05, f'CV R² = {cv_scores.mean():.3f}±{cv_scores.std():.3f}', 
        transform=ax.transAxes, ha='right', fontsize=10, style='italic')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'predictability_regression_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Cross-Visit Stability of Predictability

If predictability reflects a stable trait, per-subject accuracy at V1 should correlate with accuracy at V2 for subjects who appear in both visits.

In [ ]:
print("\n" + "="*70)
print("CROSS-VISIT STABILITY OF PREDICTABILITY")
print("="*70)

v1_df = merged_df[merged_df['visit_number'] == 1.0].copy()
v2_df = merged_df[merged_df['visit_number'] == 2.0].copy()

v1_teams = merged_df[merged_df['visit_number'] == 1.0][['subject_id', 'team']].drop_duplicates()
v2_teams = merged_df[merged_df['visit_number'] == 2.0][['subject_id', 'team']].drop_duplicates()
repeated_teams = set(v1_teams['team'].dropna()) & set(v2_teams['team'].dropna())

print(f"Teams with both V1 and V2: {len(repeated_teams)}")

v1_rep = v1_df[v1_df['team'].isin(repeated_teams)].copy()
v2_rep = v2_df[v2_df['team'].isin(repeated_teams)].copy()

print(f"V1 repeated subjects: {v1_rep['subject_id'].nunique()}, trials: {len(v1_rep)}")
print(f"V2 repeated subjects: {v2_rep['subject_id'].nunique()}, trials: {len(v2_rep)}")

if v1_rep['subject_id'].nunique() >= 5 and v2_rep['subject_id'].nunique() >= 5:
    print("\nComputing per-subject accuracy within each visit (full model, repeated teams only)...")
    acc_v1 = get_subject_prediction_accuracy(v1_rep, all_feature_cols, 'v1')
    acc_v2 = get_subject_prediction_accuracy(v2_rep, all_feature_cols, 'v2')

    acc_v1_teams = acc_v1.merge(v1_teams, on='subject_id')
    acc_v2_teams = acc_v2.merge(v2_teams, on='subject_id')
    team_v1_acc = acc_v1_teams.groupby('team')['v1_accuracy'].mean().reset_index().rename(columns={'v1_accuracy': 'v1_team_acc'})
    team_v2_acc = acc_v2_teams.groupby('team')['v2_accuracy'].mean().reset_index().rename(columns={'v2_accuracy': 'v2_team_acc'})
    stability_df = team_v1_acc.merge(team_v2_acc, on='team')

    print(f"\nTeam-level V1 vs V2 accuracy ({len(stability_df)} teams):")
    print(stability_df.to_string(index=False))

    if len(stability_df) >= 5:
        r_stab, p_stab = stats.pearsonr(stability_df['v1_team_acc'], stability_df['v2_team_acc'])
        print(f"\nV1→V2 stability correlation: r = {r_stab:.3f}, p = {p_stab:.3f}")
    else:
        print("\nToo few teams for correlation — reporting descriptive stats only.")
        r_stab, p_stab = np.nan, np.nan
        stability_df = pd.DataFrame()
else:
    print("\nInsufficient repeated subjects for cross-visit analysis.")
    r_stab, p_stab = np.nan, np.nan
    stability_df = pd.DataFrame()

In [ ]:
if len(stability_df) >= 3:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(stability_df['v1_team_acc'], stability_df['v2_team_acc'],
               c='#2E86AB', s=80, alpha=0.8, edgecolors='white', linewidths=0.5)

    for _, row in stability_df.iterrows():
        ax.annotate(str(row['team']), (row['v1_team_acc'], row['v2_team_acc']),
                    textcoords='offset points', xytext=(5, 5), fontsize=8)

    if not np.isnan(r_stab):
        z = np.polyfit(stability_df['v1_team_acc'], stability_df['v2_team_acc'], 1)
        p_fit = np.poly1d(z)
        x_line = np.linspace(stability_df['v1_team_acc'].min(), stability_df['v1_team_acc'].max(), 100)
        ax.plot(x_line, p_fit(x_line), 'r--', linewidth=2, alpha=0.7)
        title_str = f'V1→V2 Team Predictability Stability\nr = {r_stab:.3f}, p = {p_stab:.3f}'
    else:
        title_str = 'V1→V2 Team Predictability Stability\n(insufficient data for correlation)'

    ax.plot([0.4, 1], [0.4, 1], 'k--', alpha=0.3, label='y=x')
    ax.set_xlabel('V1 Mean Team Accuracy', fontweight='bold')
    ax.set_ylabel('V2 Mean Team Accuracy', fontweight='bold')
    ax.set_title(title_str, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cross_visit_stability.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Skipping stability plot — insufficient data.')

## 7. Save Results

In [ ]:
# Save all results
analysis_df.to_csv(OUTPUT_DIR / 'subject_predictability_traits.csv', index=False)
corr_df.to_csv(OUTPUT_DIR / 'trait_correlations.csv', index=False)
comparison_df.to_csv(OUTPUT_DIR / 'high_vs_low_comparison.csv', index=False)
coef_df.to_csv(OUTPUT_DIR / 'regression_coefficients.csv', index=False)

print("\n" + "="*70)
print("FILES SAVED")
print("="*70)
for f in OUTPUT_DIR.glob('*'):
    print(f"  - {f.name}")
if len(stability_df) > 0:
    stability_df.to_csv(OUTPUT_DIR / 'cross_visit_stability.csv', index=False)

## 8. Key Findings Summary

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS: INDIVIDUAL DIFFERENCES IN PREDICTABILITY")
print("="*70)

print("\n1. DISTRIBUTION OF PREDICTABILITY (full model):")
print(f"   - Range: {analysis_df['full_model_accuracy'].min():.1%} to {analysis_df['full_model_accuracy'].max():.1%}")
print(f"   - Mean: {analysis_df['full_model_accuracy'].mean():.1%} ± {analysis_df['full_model_accuracy'].std():.1%}")

print("\n2. TOP CORRELATES OF PREDICTABILITY:")
for _, row in corr_df.head(3).iterrows():
    direction = 'positively' if row['r'] > 0 else 'negatively'
    print(f"   - {row['trait']}: r = {row['r']:.3f} (p = {row['p']:.2e})")

print("\n3. HIGH vs LOW PREDICTABILITY SUBJECTS:")
sig_diffs = comparison_df[comparison_df['p_value'] < 0.05].sort_values('p_value')
if len(sig_diffs) > 0:
    for _, row in sig_diffs.head(3).iterrows():
        print(f"   - {row['trait']}: d = {row['cohens_d']:.2f}")
else:
    print("   - No significant differences found")

print("\n4. CROSS-VISIT STABILITY:")
if not np.isnan(r_stab):
    print(f"   - Team-level V1→V2 accuracy correlation: r = {r_stab:.3f}, p = {p_stab:.3f}")
    stability_interp = "consistent across sessions (trait-like)" if p_stab < 0.05 else "not significantly stable across sessions"
    print(f"   - Predictability is {stability_interp}")
else:
    print("   - Cross-visit stability: insufficient paired data")

print("\n5. AMBIGUITY AVERSION & GAZE COUPLING:")
aversion_corr = corr_df[corr_df['trait'] == 'aversion_score']
coupling_corr = corr_df[corr_df['trait'] == 'gaze_coupling_score']
if len(aversion_corr) > 0:
    row = aversion_corr.iloc[0]
    print(f"   - Aversion score: r = {row['r']:.3f} (p = {row['p']:.2e})")
if len(coupling_corr) > 0:
    row = coupling_corr.iloc[0]
    print(f"   - Gaze coupling score: r = {row['r']:.3f} (p = {row['p']:.2e})")

print(f"\n{'='*70}")
print(f"Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)